# llmlip + ASE チュートリアル

このチュートリアルでは、Rust ネイティブ MLIP 推論エンジン **llmlip** を
[ASE (Atomic Simulation Environment)](https://wiki.fysik.dtu.dk/ase/) から
利用する方法を解説します。

ASE は原子・分子シミュレーションのための Python ライブラリで、
構造の構築、可視化、構造最適化、分子動力学 (MD) などの機能を持ちます。
`LlmlipCalculator` を使えば、これらの ASE 機能を llmlip の高速推論で駆動できます。

## 目次

1. [セットアップ](#1-セットアップ)
2. [基本的な使い方: エネルギーと力の計算](#2-基本的な使い方-エネルギーと力の計算)
3. [分子の構築と計算](#3-分子の構築と計算)
4. [構造最適化 (Geometry Optimization)](#4-構造最適化-geometry-optimization)
5. [振動解析 (Vibration Analysis)](#5-振動解析-vibration-analysis)
6. [分子動力学 (Molecular Dynamics)](#6-分子動力学-molecular-dynamics)
7. [周期系: バルク結晶の計算](#7-周期系-バルク結晶の計算)
8. [複数分子の一括計算](#8-複数分子の一括計算)
9. [まとめ](#9-まとめ)

## 1. セットアップ

### 前提条件

- llmlip バイナリがビルド済みであること
- Python 3.9 以上
- ASE がインストール済みであること

### llmlip のビルド

```bash
cd /path/to/e3nn-core-rs
cargo build --release -p llmlip-cli
```

### Python 依存パッケージのインストール

```bash
pip install ase numpy
```

In [ ]:
import sys
from pathlib import Path

# llmlip パッケージへのパスを追加
# (pip install / maturin develop していない場合のみ必要。CLI バックエンドは
#  ネイティブモジュール無しでも動作します)
REPO_ROOT = Path("../..").resolve()  # リポジトリルートへの相対パス (環境に応じて調整)
sys.path.insert(0, str(REPO_ROOT / "crates" / "llmlip-py" / "python"))

import numpy as np
import ase
from ase import Atoms
from ase.build import molecule, bulk

print(f"ASE version: {ase.__version__}")
print(f"NumPy version: {np.__version__}")

### LlmlipCalculator の初期化

`LlmlipCalculator` は 2 つのパラメータで初期化します:

| パラメータ | 説明 |
|---|---|
| `model_path` | `.e3rs` モデルディレクトリへのパス |
| `binary` | `llmlip` バイナリへのパス (省略時は自動検出) |

バイナリの自動検出の優先順位:
1. `binary` 引数で直接指定
2. 環境変数 `LLMLIP_BINARY`
3. `$PATH` 上の `llmlip`
4. `<リポジトリルート>/target/release/llmlip`

In [ ]:
from llmlip.ase import LlmlipCalculator

# MACE-Osaka26 モデルを使用
MODEL_PATH = REPO_ROOT / "fixtures" / "mace" / "v0_3_16_4d2da09" / "models" / "mace_osaka26_runtime.e3rs"

calc = LlmlipCalculator(
    model_path=str(MODEL_PATH),
    # binary="path/to/llmlip",  # 自動検出できない場合は明示的に指定
)

print(f"Model: {MODEL_PATH.name}")
print(f"Backend: {calc.backend_name}")

---

## 2. 基本的な使い方: エネルギーと力の計算

ASE の標準的な使い方で、エネルギーと力を計算できます。

In [ ]:
# 水分子を構築
water = molecule("H2O")
water.calc = calc

# エネルギー (eV)
energy = water.get_potential_energy()
print(f"Energy: {energy:.6f} eV")

# 力 (eV/angstrom)
forces = water.get_forces()
print(f"\nForces (eV/A):")
for symbol, force in zip(water.get_chemical_symbols(), forces):
    print(f"  {symbol}: [{force[0]:+.6f}, {force[1]:+.6f}, {force[2]:+.6f}]")

---

## 3. 分子の構築と計算

ASE の `molecule()` 関数で、よく知られた分子を簡単に構築できます。
いくつかの分子でエネルギーを比較してみましょう。

In [ ]:
# さまざまな分子のエネルギーを計算
molecules = {
    "H2": "水素 (H2)",
    "H2O": "水 (H2O)",
    "NH3": "アンモニア (NH3)",
    "CH4": "メタン (CH4)",
    "C2H6": "エタン (C2H6)",
    "CH3OH": "メタノール (CH3OH)",
}

print(f"{\"分子\":<20} {\"原子数\":>6} {\"エネルギー (eV)\":>18} {\"最大力 (eV/A)\":>16}")
print("-" * 65)

results = {}
for name, label in molecules.items():
    atoms = molecule(name)
    atoms.calc = LlmlipCalculator(model_path=str(MODEL_PATH))
    e = atoms.get_potential_energy()
    f = atoms.get_forces()
    fmax = np.max(np.linalg.norm(f, axis=1))
    results[name] = {"energy": e, "fmax": fmax, "natoms": len(atoms)}
    print(f"{label:<20} {len(atoms):>6d} {e:>18.6f} {fmax:>16.6f}")

### 手動で構造を定義する

ASE の `Atoms` クラスを使って、任意の構造を定義できます。

In [ ]:
# CO2 分子を手動で構築 (直線型)
co2 = Atoms(
    symbols="OCO",
    positions=[
        [0.0, 0.0, -1.16],
        [0.0, 0.0,  0.00],
        [0.0, 0.0,  1.16],
    ],
)
co2.calc = LlmlipCalculator(model_path=str(MODEL_PATH))

energy = co2.get_potential_energy()
forces = co2.get_forces()
print(f"CO2 Energy: {energy:.6f} eV")
print(f"Forces (eV/A):")
for s, f in zip(co2.get_chemical_symbols(), forces):
    print(f"  {s}: [{f[0]:+.8f}, {f[1]:+.8f}, {f[2]:+.8f}]")

---

## 4. 構造最適化 (Geometry Optimization)

ASE の最適化アルゴリズムと組み合わせて、構造最適化 (エネルギー最小化) を実行できます。

ここでは水分子の構造を少し歪ませてから最適化し、安定構造を得ます。

In [ ]:
from ase.optimize import BFGS

# 水分子を構築し、位置を少し歪ませる
water_distorted = molecule("H2O")
water_distorted.positions[1] += [0.1, -0.05, 0.08]  # H原子を移動
water_distorted.positions[2] += [-0.08, 0.1, -0.05]  # もう一つのH原子を移動

water_distorted.calc = LlmlipCalculator(model_path=str(MODEL_PATH))

# 最適化前のエネルギーと最大力
e_before = water_distorted.get_potential_energy()
f_before = water_distorted.get_forces()
fmax_before = np.max(np.linalg.norm(f_before, axis=1))
print(f"最適化前:")
print(f"  エネルギー: {e_before:.6f} eV")
print(f"  最大力:     {fmax_before:.6f} eV/A")
print()

In [ ]:
# BFGS 法による構造最適化
opt = BFGS(water_distorted, logfile="-")  # logfile="-" で標準出力にログ
opt.run(fmax=0.01)  # 最大力が 0.01 eV/A 以下になるまで最適化

# 最適化後の結果
e_after = water_distorted.get_potential_energy()
f_after = water_distorted.get_forces()
fmax_after = np.max(np.linalg.norm(f_after, axis=1))
print(f"\n最適化後:")
print(f"  エネルギー: {e_after:.6f} eV")
print(f"  最大力:     {fmax_after:.6f} eV/A")
print(f"  ステップ数: {opt.nsteps}")
print(f"  エネルギー変化: {e_after - e_before:.6f} eV")

In [ ]:
# 最適化後の構造を確認
print("最適化後の座標 (A):")
for s, p in zip(water_distorted.get_chemical_symbols(), water_distorted.positions):
    print(f"  {s}: [{p[0]:+.6f}, {p[1]:+.6f}, {p[2]:+.6f}]")

# O-H 結合長を計算
d_oh1 = water_distorted.get_distance(0, 1)
d_oh2 = water_distorted.get_distance(0, 2)
angle = water_distorted.get_angle(1, 0, 2)
print(f"\nO-H 結合長: {d_oh1:.4f} A, {d_oh2:.4f} A")
print(f"H-O-H 結合角: {angle:.2f} deg")

---

## 5. 振動解析 (Vibration Analysis)

最適化された構造に対して振動解析を行い、振動数（波数）を計算できます。
ASE は有限差分法でヘッセ行列を数値的に計算します。

In [ ]:
from ase.vibrations import Vibrations

# まず構造を最適化
water_vib = molecule("H2O")
water_vib.calc = LlmlipCalculator(model_path=str(MODEL_PATH))

opt = BFGS(water_vib, logfile=None)
opt.run(fmax=0.001)  # 振動解析のためにしっかり最適化
print(f"最適化完了: {opt.nsteps} ステップ")
print(f"最適化後エネルギー: {water_vib.get_potential_energy():.6f} eV")

In [ ]:
# 振動解析 (有限差分法)
vib = Vibrations(water_vib, name="vib_h2o")
vib.run()

# 振動数の表示
print("振動解析結果:")
vib.summary()

# クリーンアップ
vib.clean()

水分子の実験値との比較:

| モード | 実験値 (cm$^{-1}$) | 説明 |
|---|---|---|
| 対称伸縮 | 3657 | O-H 結合の対称的な伸縮 |
| 変角 | 1595 | H-O-H 角度の変化 |
| 非対称伸縮 | 3756 | O-H 結合の非対称的な伸縮 |

MLIP は DFT データで学習されているため、実験値とは多少ずれることがありますが、
定性的な傾向（伸縮振動 > 変角振動）は再現されるはずです。

---

## 6. 分子動力学 (Molecular Dynamics)

ASE の MD エンジンと組み合わせて、分子動力学シミュレーションを実行できます。

ここではエタン分子の NVE (ミクロカノニカル) MD を短時間実行します。

In [ ]:
from ase.md.verlet import VelocityVerlet
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase import units

# エタン分子を構築
ethane = molecule("C2H6")
ethane.calc = LlmlipCalculator(model_path=str(MODEL_PATH))

# 初期速度を設定 (300 K)
MaxwellBoltzmannDistribution(ethane, temperature_K=300)

# NVE 分子動力学
dt = 0.5 * units.fs  # タイムステップ: 0.5 fs
dyn = VelocityVerlet(ethane, timestep=dt)

# エネルギーの履歴を記録
energies = []
temperatures = []

def record_state():
    """各ステップのエネルギーと温度を記録するコールバック。"""
    e_pot = ethane.get_potential_energy()
    e_kin = ethane.get_kinetic_energy()
    temp = e_kin / (1.5 * units.kB * len(ethane))
    energies.append({"potential": e_pot, "kinetic": e_kin, "total": e_pot + e_kin})
    temperatures.append(temp)

# 初期状態を記録
record_state()

# 20 ステップ実行 (10 fs)
n_steps = 20
print(f"MD 実行: {n_steps} ステップ (dt = 0.5 fs, 合計 {n_steps * 0.5:.1f} fs)")
print(f"{"Step":>6} {"E_pot (eV)":>12} {"E_kin (eV)":>12} {"E_tot (eV)":>12} {"T (K)":>8}")
print("-" * 55)

for step in range(n_steps):
    dyn.run(steps=1)
    record_state()
    if (step + 1) % 5 == 0 or step == 0:
        e = energies[-1]
        print(f"{step + 1:>6d} {e["potential"]:>12.6f} {e["kinetic"]:>12.6f} "
              f"{e["total"]:>12.6f} {temperatures[-1]:>8.1f}")

In [ ]:
# エネルギー保存の確認
e_total = [e["total"] for e in energies]
e_drift = max(e_total) - min(e_total)
print(f"\n全エネルギーのドリフト: {e_drift:.6e} eV")
print(f"全エネルギーの標準偏差: {np.std(e_total):.6e} eV")
print(f"温度範囲: {min(temperatures):.1f} - {max(temperatures):.1f} K")

NVE アンサンブルでは全エネルギー (ポテンシャル + 運動エネルギー) が保存されます。
ドリフトが小さいほど、力場が保存力 ($F = -\nabla E$) を正しく再現していることを
意味します。

---

## 7. 周期系: バルク結晶の計算

ASE の `bulk()` 関数で結晶構造を構築し、エネルギー・力・応力を計算できます。

**注意**: 応力テンソルの計算には完全な周期境界条件 (`pbc=[True, True, True]`) が
必要です。

In [ ]:
# 銅 (FCC) のバルク結晶
cu_bulk = bulk("Cu", crystalstructure="fcc", a=3.6)
cu_bulk.calc = LlmlipCalculator(model_path=str(MODEL_PATH))

energy = cu_bulk.get_potential_energy()
forces = cu_bulk.get_forces()
stress = cu_bulk.get_stress()  # Voigt 形式: [xx, yy, zz, yz, xz, xy]

print(f"Cu (FCC) bulk:")
print(f"  原子数: {len(cu_bulk)}")
print(f"  格子定数: {cu_bulk.cell.lengths()[0]:.4f} A")
print(f"  エネルギー: {energy:.6f} eV")
print(f"  1原子あたり: {energy / len(cu_bulk):.6f} eV/atom")
print(f"  最大力: {np.max(np.abs(forces)):.2e} eV/A")
print(f"  応力 (Voigt, eV/A^3): {stress}")

In [ ]:
# 格子定数を変えてエネルギーカーブを描く
a_values = np.linspace(3.3, 4.0, 15)
energies_per_atom = []

for a in a_values:
    cu = bulk("Cu", crystalstructure="fcc", a=a)
    cu.calc = LlmlipCalculator(model_path=str(MODEL_PATH))
    e = cu.get_potential_energy() / len(cu)
    energies_per_atom.append(e)

# 最小エネルギーの格子定数を見つける
idx_min = np.argmin(energies_per_atom)
a_opt = a_values[idx_min]
e_min = energies_per_atom[idx_min]

print(f"Cu FCC 格子定数 vs エネルギー:")
print(f"  最適格子定数: {a_opt:.3f} A")
print(f"  最小エネルギー: {e_min:.6f} eV/atom")
print(f"  (実験値: 3.615 A)")

### ASE の EOS フィッティング

ASE には状態方程式 (Equation of State) フィッティングの機能もあります。
格子定数と体積弾性率を求めることができます。

In [ ]:
from ase.eos import EquationOfState
from ase.units import GPa

# 体積とエネルギーのリストを作成
volumes = []
eos_energies = []

for a in np.linspace(3.4, 3.9, 11):
    cu = bulk("Cu", crystalstructure="fcc", a=a)
    cu.calc = LlmlipCalculator(model_path=str(MODEL_PATH))
    volumes.append(cu.get_volume())
    eos_energies.append(cu.get_potential_energy())

# Birch-Murnaghan 状態方程式でフィッティング
eos = EquationOfState(volumes, eos_energies, eos="birchmurnaghan")
v0, e0, B = eos.fit()

# 結果の表示
a_eos = (4 * v0) ** (1/3)  # FCC: 4 atoms per unit cell -> v0 = a^3/4
B_GPa = B / GPa
print(f"Birch-Murnaghan フィッティング結果:")
print(f"  平衡体積: {v0:.4f} A^3")
print(f"  平衡格子定数: {a_eos:.4f} A")
print(f"  体積弾性率: {B_GPa:.1f} GPa")
print(f"  (Cu 実験値: a = 3.615 A, B = 140 GPa)")

---

## 8. 複数分子の一括計算

Python のリスト内包表記を使って、複数の分子を一括で計算できます。

In [ ]:
# 有機分子系列のエネルギーを計算
organic_series = [
    ("CH4", "メタン"),
    ("C2H6", "エタン"),
    ("C2H4", "エチレン"),
    ("C2H2", "アセチレン"),
]

print(f"{\"分子\":<14} {\"化学式\":<8} {\"原子数\":>6} {\"E (eV)\":>12} {\"E/atom (eV)\":>12}")
print("-" * 57)

for formula, name in organic_series:
    atoms = molecule(formula)
    atoms.calc = LlmlipCalculator(model_path=str(MODEL_PATH))
    e = atoms.get_potential_energy()
    n = len(atoms)
    print(f"{name:<14} {formula:<8} {n:>6d} {e:>12.4f} {e/n:>12.4f}")

### extxyz ファイルの読み書き

ASE の I/O 機能を使えば、構造ファイルの読み書きも簡単です。

In [ ]:
from ase.io import read, write

# llmlip チュートリアル付属の分子を読み込む
tutorial_molecules_dir = REPO_ROOT / "docs" / "tutorials" / "molecules"

if tutorial_molecules_dir.exists():
    for extxyz_file in sorted(tutorial_molecules_dir.glob("*.extxyz")):
        atoms = read(str(extxyz_file))
        atoms.calc = LlmlipCalculator(model_path=str(MODEL_PATH))
        e = atoms.get_potential_energy()
        print(f"{extxyz_file.stem:<12} ({len(atoms):>2d} atoms): E = {e:.6f} eV")
else:
    print(f"Tutorial molecules directory not found: {tutorial_molecules_dir}")

In [ ]:
# 計算結果を extxyz に保存 (エネルギーと力を含む)
import tempfile, os

water_save = molecule("H2O")
water_save.calc = LlmlipCalculator(model_path=str(MODEL_PATH))

# エネルギーと力を計算
water_save.get_potential_energy()
water_save.get_forces()

# 一時ファイルに保存して内容を確認
with tempfile.NamedTemporaryFile(suffix=".extxyz", mode="w", delete=False) as f:
    tmppath = f.name

write(tmppath, water_save, format="extxyz")
print("保存された extxyz ファイルの内容:")
with open(tmppath) as f:
    print(f.read())
os.unlink(tmppath)

---

## 9. まとめ

このチュートリアルでは、`LlmlipCalculator` を使って llmlip (Rust MLIP 推論エンジン) を
ASE から利用する方法を学びました。

### できること

| 機能 | ASE メソッド | 必要な計算 |
|---|---|---|
| エネルギー計算 | `atoms.get_potential_energy()` | energy |
| 力の計算 | `atoms.get_forces()` | forces |
| 応力テンソル | `atoms.get_stress()` | stress (PBC 必須) |
| 構造最適化 | `BFGS(atoms).run(fmax=0.01)` | energy + forces |
| 振動解析 | `Vibrations(atoms)` | forces (有限差分) |
| 分子動力学 | `VelocityVerlet(atoms, dt)` | forces |
| 状態方程式 | `EquationOfState(V, E)` | energy |

### LlmlipCalculator の特徴

- **Python 不要の推論**: 内部で Rust バイナリ (`llmlip`) を呼び出すため、
  PyTorch や MACE Python パッケージのインストールが不要
- **高い数値精度**: MACE 本家 (Python) と 1e-10 eV 以下の差で一致
- **高速**: シングルスレッドで約 2.9 倍、マルチスレッドで最大 3.3 倍高速
- **対応モデル**: MACE-Osaka26 (97 元素)、MACE-MP0b、NeQUIP-OAM-L など

### クイックリファレンス

```python
from llmlip.ase import LlmlipCalculator

calc = LlmlipCalculator(
    model_path="path/to/model.e3rs",
    binary="path/to/llmlip",  # 省略可能 (自動検出)
)
atoms.calc = calc
```

### 参考リンク

- [ASE ドキュメント](https://wiki.fysik.dtu.dk/ase/)
- [e3nn-core-rs リポジトリ](https://github.com/mizukami-group/e3nn-core-rs)
- [MACE-Osaka26](https://github.com/qiqb-osaka/mace-osaka26)
- [MACE-Osaka26 Rust チュートリアル](mace_osaka26_tutorial_ja.md)